# IHGAMP - Step 5: External Validation

Validate on external cohorts:
- CPTAC (LUAD, LUSC, HNSCC, UCEC)
- PTRC-HGSOC
- OBR-Bevacizumab

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models import HRDPredictor
from src.evaluation import evaluate_with_calibration

OUTPUT_DIR = Path('./outputs')
RESULTS_DIR = OUTPUT_DIR / 'results' / 'external'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load trained model
model = HRDPredictor.load(str(OUTPUT_DIR / 'models' / 'hrd_predictor.joblib'))
print('Model loaded')

In [ ]:
# Define external cohorts - UPDATE THESE PATHS
EXTERNAL_COHORTS = {
    'CPTAC-LUAD': {
        'embeddings': 'path/to/cptac_luad_embeddings.parquet',
        'labels': 'path/to/cptac_luad_labels.csv',
        'task': 'HRD'
    },
    'CPTAC-LUSC': {
        'embeddings': 'path/to/cptac_lusc_embeddings.parquet',
        'labels': 'path/to/cptac_lusc_labels.csv',
        'task': 'HRD'
    },
    'PTRC-HGSOC': {
        'embeddings': 'path/to/ptrc_embeddings.parquet',
        'labels': 'path/to/ptrc_labels.csv',
        'task': 'Platinum Response'
    },
}

In [ ]:
# Evaluate external cohorts
external_results = []

for cohort_name, config in EXTERNAL_COHORTS.items():
    print(f'\n=== {cohort_name} ===')
    try:
        embeddings = pd.read_parquet(config['embeddings'])
        labels = pd.read_csv(config['labels'])
        df = embeddings.merge(labels, on='patient', how='inner')
        
        z_cols = [c for c in df.columns if c.startswith('z')]
        X = df[z_cols].values.astype(np.float32)
        y = df['label'].values
        
        pred = model.predict_proba(X)
        results = evaluate_with_calibration(y, pred, cohort_name=cohort_name)
        external_results.append(results['primary_metrics'])
        
        m = results['primary_metrics'].iloc[0]
        print(f"  n={m['n']}, AUC={m['auc']:.3f} ({m['auc_ci_low']:.3f}-{m['auc_ci_high']:.3f})")
    except FileNotFoundError:
        print(f'  Files not found - update paths')
    except Exception as e:
        print(f'  Error: {e}')

In [ ]:
# Combined results table
if external_results:
    external_table = pd.concat(external_results, ignore_index=True)
    external_table = external_table[external_table['variant'] == 'Uncalibrated']
    print('\n=== External Validation Summary ===')
    print(external_table[['cohort', 'n', 'auc', 'ap']].to_string(index=False))
    external_table.to_csv(RESULTS_DIR / 'external_validation.csv', index=False)

In [ ]:
# Forest plot
if external_results:
    fig, ax = plt.subplots(figsize=(8, len(external_table)*0.8+1))
    cohorts = external_table['cohort'].values
    aucs = external_table['auc'].values
    ci_lo = external_table['auc_ci_low'].values
    ci_hi = external_table['auc_ci_high'].values
    
    y_pos = np.arange(len(cohorts))
    ax.errorbar(aucs, y_pos, xerr=[aucs-ci_lo, ci_hi-aucs], fmt='o', capsize=4, markersize=8)
    ax.axvline(x=0.5, color='gray', ls='--', alpha=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(cohorts)
    ax.set_xlabel('AUC (95% CI)')
    ax.set_xlim([0.3, 1.0])
    ax.set_title('External Validation')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'forest_plot.png', dpi=300, bbox_inches='tight')
    plt.show()